# Readme

This notebook is composed of two parts:

1. Data Ingestion Pipeline: This extracts data from two online data sources, cleans it, and loads it into a SQLite database.

  **N.B.** Data will be loaded to local .csv files. You will then be prompted whether you want to upload these to the SQLite database. In the first instance this needs to be done.

2. Exploratory Data Analysis: Uses the pre-preprepared SQLite database for data, manipulates the data, and presents data analytics.

**N.B.** The first section must be run first for the second section to work.

# Data Ingestion Pipeline

## [Contents](#contents)

- [1. Overview](#1-overview)

- [2. Environment Configuration](#2-environment-configuration)

- [3. Crop Data](#3-crop-data)
  - [3.1 Download the .ods object from the resource URL](#31-download-the-ods-object-from-the-resource-url)
  - [3.2 Create list of all sheets in the .ods workbook](#32-create-list-of-all-sheets-in-the-ods-workbook)
  - [3.3 3.3 Parse each sheet into separate raw data .csv](#33-parse-each-sheet-into-separate-raw-data-csv)
  - [3.4 Further cleanse raw .csv for loading into Pandas dataframe](#34-further-cleanse-raw-csv-for-loading-into-pandas-dataframe)
  - [3.5 Parse cleansed .csv into Pandas for further wrangling and SQLite loading](#35-parse-cleansed-csv-into-pandas-for-further-wrangling-and-sqlite-loading)

- [4. Weather Data](#4-weather-data)
  - [4.1 Resource URL](#41-resource-url)
  - [4.2 Weather types and Areas](#42-weather-types-and-areas)
  - [4.3 Parse weather data into local .csv](#43-parse-weather-data-into-local-csv)
  - [4.4 Load into SQLite](#44-load-into-sqlite)

## [1. Overview](#contents)

This notebook is concerned with the extraction, transformation and loading of publicly available data into a database to provide datasets for downstream analytical query.

Two datasets are ingested:

- Weather data from the UK Meteorological service.
- Crop yield data from the UK Department for Environment, Farming and Rural Affairs (DEFRA).

Each dataset will be first downloaded to local directories in the form of .csv. This will allow for continued processing without having need of an internet connection.

Once raw data has been downloaded into raw .csv files, these will be parsed into Pandas Dataframes for wrangling into formats suitable for writing to a simple SQLite database.

The resulting database will then be available for the next analytical jupyter notebook.

## [2. Environment Configuration](#contents)

This needs to be run at the beginning of any new session to ensure the correct libraries are installed.

In [ ]:
# For calling data repository URL
import requests
import io

# for file and directory manipulation
import os

# For manipulating data with Pandas
import pandas as pd

# using the sqlite3 database
import sqlite3

# Ensure the odfpy library is installed
%pip install odfpy

## [3. Crop Data](#contents)

### 3.1 Download the .ods object from the resource URL

In [ ]:
# N.B. Need to ensure odfpy is installed in the environment. This allows the ODS binary to be parsed.

# Creates a ByteIO Class Object

url = "https://assets.publishing.service.gov.uk/media/68e778c1e5f463a62cb98588/UK-cops-webseries-251009a.ods"
try:
    response = requests.get(url, timeout=30)
    response.raise_for_status()  # Raises for 4xx/5xx codes.
    buffer_object = io.BytesIO(response.content)
    print(f"Download succeeded: {len(response.content)} bytes")
except requests.exceptions.RequestException as e:
    buffer_object = None
    print(f"ERROR: Unable to download ODS file from {url}: {e}")
except Exception as e:
    buffer_object = None
    print(f"Unexpected error while creating buffer_object: {e}")

### 3.2 Create list of all sheets in the .ods workbook

In [ ]:
# https://pandas.pydata.org/docs/dev/reference/api/pandas.ExcelFile.sheet_names.html
# this outputs an ExcelFile Class Object

list_odf_sheets = []

if buffer_object is None:
    raise RuntimeError("buffer_object is not available; cannot read workbook")

try:
    crops_file = pd.ExcelFile(path_or_buffer=buffer_object, engine="odf")
    list_odf_sheets = list(crops_file.sheet_names)
    print(f"Sheets found: {len(list_odf_sheets)}")
except Exception as e:
    list_odf_sheets = []
    print(f"ERROR: Unable to read sheets from ODS workbook: {e}")

list_odf_sheets


### 3.3 Parse each sheet into separate raw data .csv

For each sheet with suffix 'Region' parse into from **buffer_object** into dataframes and output as local .csv files.

Objects:
- label: **raw_wheat.csv**, object type: **.csv**
- label: **raw_winter_barley.csv**, object type: **.csv**
- label: **raw_spring_barley.csv**, object type: **.csv**
- label: **raw_total_barley.csv**, object type: **.csv**
- label: **raw_oats.csv**, object type: **.csv**
- label: **raw_osr.csv**, object type: **.csv**

Locally saving the file allows decoupling the need for network calls, allowing for faster I/O operations. This also allows me to continue development whilst internet connection was not availalble, e.g. whilst travelling on the train.

Exporting locally to .csv files does come at the cost of storage. This can be mitigated by deleting the files once processing is completed, retaining only the required data for reference. However, the size of data is very small and easily maintained on even the most limited of systems.

In [ ]:
# Define the prefix
prefix = 'Regional'

# Ensure that the directory exists before writing files
os.makedirs('data/crops/raw', exist_ok=True)

if not list_odf_sheets:
    print("WARNING: No sheets detected. Skipping raw CSV export step.")

# Use for loop to list through list of sheet names, parse sheet into dataframe, and output locally as .csv file.
for name in list_odf_sheets:
    if name.startswith(prefix):
        try:
            df_sheet = pd.ExcelFile(path_or_buffer=buffer_object, engine="odf").parse(name)
            output_path = f'data/crops/raw/{name}.csv'
            df_sheet.to_csv(output_path, index=False)
            print(f"Exported {name} -> {output_path}")
        except Exception as e:
            print(f"ERROR: failed to export sheet {name}: {e}")


### 3.4 Further cleanse raw .csv for loading into Pandas dataframe

- For each sheet CSV:
  - load into dataframe.
  - crop to new dataframe with relevant rows and columns.
  - Reset column headers and index to years.
- Save as csv in "yeilds" directory.

#### 3.4.1 List of target .csv files

In [ ]:
crop_file_names = [
			'Regional_oats',
			'Regional_OSR',
			'Regional_spring_barley',
			'Regional_total_barley',
			'Regional_wheat',
			'Regional_winter_barley'
]

#### 3.4.2 Loop through each sheet and output formatted .csv

This is a consolidated process for apply this across all the crop files at once.

In [ ]:
# parameterise start and end rows for yield section.
yield_headers_start = 17
yield_rows = 13

for crop in crop_file_names:
    try:
        file_path = f'data/crops/raw/{crop}.csv'
        df_raw = pd.read_csv(file_path, header=yield_headers_start, nrows=yield_rows)

        # Drop unwanted columns.
        unwanted_columns = ["% change 2025/2024", "2025 confidence interval", "Unnamed: 30", "5 year average"]
        df_crop = df_raw.drop(columns=[col for col in unwanted_columns if col in df_raw.columns], errors='ignore')

        # Rename columns for consistency.
        df_crop.rename(columns={
            'Yields (1)': 'Year',
            '2022(1)': '2022',
            '2022(a)': '2022',
        }, inplace=True)

        # Transpose dataframe with regions as columns and years as rows.
        df_long = df_crop.transpose()

        # Make the first row, row index 0, the column names.
        if df_long.empty:
            raise ValueError(f"Extracted table empty for crop {crop}")

        df_long.columns = df_long.iloc[0]

        # Reset the dataframe to start without the first row that has now become the column names.
        df_long = df_long[1:].copy()

        # Clear the column heading name.
        df_long.columns.name = None

        # Reset the index to make years a regular column
        df_long.reset_index(inplace=True)

        # Rename new column for clarity (the index becomes a column named 'index' by default)
        df_long.rename(columns={'index': 'Year'}, inplace=True)

        # Convert data types
        for column in df_long.columns:
            if column == 'Year':
                df_long[column] = pd.to_numeric(df_long[column], errors='coerce').astype('Int64')
            else:
                df_long[column] = pd.to_numeric(df_long[column], errors='coerce').astype(float)

        # Save to "yields" directory as transformed csv file.
        os.makedirs('data/crops/yields/', exist_ok=True)

        name = f'yields_{crop[9:]}'
        df_long.to_csv(f'data/crops/yields/{name}.csv', index=False)
        print(f"Processed and saved {name}.csv")

    except FileNotFoundError:
        print(f"ERROR: raw file not found for crop {crop}: {file_path}")
    except Exception as e:
        print(f"ERROR processing crop {crop}: {e}")


### 3.5 Parse cleansed .csv into Pandas for further wrangling and SQLite loading

#### 3.5.1 Prerequisites for SQLite

In [ ]:
os.makedirs('data', exist_ok=True)
project_db  = 'data/project_db.db'

#### 3.5.2 Load all formatted .csv into SQLite database

In [ ]:
# Get all file names to loop through.
yield_file_list: list = os.listdir('data/crops/yields')

print(yield_file_list)

In [ ]:
for yield_file in yield_file_list:

	# Parse file to dataframe.
	df_yield = pd.read_csv(f'data/crops/yields/{yield_file}')

	# Truncate file name for table name.
	table_name = yield_file[:-4]

	# open connection to database.
	conn = sqlite3.connect(project_db)

	try:
		# Load dataframe to sqlite.
		df_yield.to_sql(
			name=table_name,
			con=conn,
			if_exists='replace',
			index=False
		)

		print(f'Table {table_name} updated')

	except Exception as e:
		print(f"Error: {(str(e))}")
	
	finally:
		# Close connection
		conn.close()


#### 3.5.3 Test database query

In [ ]:
conn = sqlite3.connect(project_db)

df_db_sp_yields = pd.read_sql('SELECT * FROM yields_spring_barley', conn)
conn.close()

df_db_sp_yields.head()

## [4. Weather Data](#contents)

### 4.1 Resource URL

For the purposes of retrieving weather data by year, to select the data to download, two parameters are passed: the **Weather Type**, and the **Area**. This information was manually extracted from the html of the website: <https://www.metoffice.gov.uk/research/climate/maps-and-data/uk-and-regional-series>

The for requests follows the format <https://www.metoffice.gov.uk/pub/data/weather/uk/climate/datasets/{weather}/date/{region}.txt>. Note the **weather** and **area** parameters in curly braces. These are used to determine which specific dataset is requested.

### 4.2 Weather types and Areas

Data retrieved from the url specifies a weather type and region. These can be captured from inspecting the website.

#### 4.2.1 Weather data

These are the parameters used.

|Parameter|Label|
|---------|-----|
|Max temp|Tmax|
|Min temp|Tmin|
|Mean temp|Tmean|
|Sunshine|Sunshine|
|Rainfall|Rainfall|
|Rain days ≥1.0mm|Raindays1mm|
|Days of air frost|AirFrost|

This "weather_types" list  can be used to iterate over to retrieve all weather types.

In [ ]:
weather_types = ["Tmax", "Tmin", "Tmean", "Sunshine", "Rainfall", "Raindays1mm", "AirFrost"]

print(weather_types)

#### 4.2.2 Regions

These Regions are how the data for each weather type is aggregated and segmented.

|ID|Region|Label|
|--|------|-----|
|1|UK|UK|
|2|England|England|
|3|Wales|Wales|
|4|Scotland|Scotland|
|5|Northern Ireland|Northern_Ireland|
|6|England & Wales|England_and_Wales|
|7|England N|England_N|
|8|Englan S|Englan_S|
|9|Scotland N|Scotland_N|
|10|Scotland E|Scotland_E|
|11|Scotland W|Scotland_W|
|12|England E & NE|England_E_and_NE|
|13|England NW/Wales N|England_NW_and_N_Wales|
|14|Midlands|Midlands|
|15|East Anglia|East_Anglia|
|16|England SW/Wales S|England_SW_and_S_Wales|
|17|England SE/Central S|England_SE_and_Central_S|

This "areas" list can be used to iterate to request each of the areas in turn for each of the weather types previously noted.

In [ ]:
areas = ["UK",
		"England",
		"Wales",
		"Scotland",
		"Northern_Ireland",
		"England_and_Wales",
		"England_N",
		"England_S",
		"Scotland_N",
		"Scotland_E",
		"Scotland_W",
		"England_E_and_NE",
		"England_NW_and_N_Wales",
		"Midlands",
		"East_Anglia",
		"England_SW_and_S_Wales",
		"England_SE_and_Central_S"]

print (areas)

### 4.3 Parse weather data into local .csv

Rather than downloading each of file for each region for each parameter, it would be better to directly retrieve the data from the API or URL.

The URL clearly indicates in plain text the Region and Parameter labels. I can extract these and inject them into the URL string used to request the data.

In [ ]:
# For each Weather Type, for each Area, request data from the met office website.
for weather in weather_types:
	for area in areas:

		# Ensure that the directory exists before writing files
		os.makedirs(f'data/weather/{weather}', exist_ok=True)

		# Get data with parametrised URL.
		url= f"https://www.metoffice.gov.uk/pub/data/weather/uk/climate/datasets/{weather}/date/{area}.txt"
		response = requests.get(url)
		print (response.text)

		# Read with flexible whitespace handling.
		try:
			df = pd.read_csv(url, 
							 sep='\s+',					# Handle variable spacing
							 skipinitialspace=True,		# Skip leading spaces
							 skip_blank_lines=True,		# Skip empty lines
							 skiprows=5)				# Skip the first 5 lines of metadata

			# Write to CSV with a filename that includes the weather type and area.
			df.to_csv(f"data/weather/{weather}/{areas}_{weather}.csv", index=False)

		# Catch any exceptions.
		except Exception as e:
			print(f"Error: {str(e)}")

### 4.4 Load into SQLite

#### 4.4.1 Prerequisites for SQLite

In [ ]:
os.makedirs('data', exist_ok=True)
project_db  = 'data/project_db.db'

#### 4.4.2 Load .csv into SQLite database

For offline loading, the downloaded .csv can be loaded into the SQLite database.

In [ ]:
# function to loop through both lists and retrieve data from url.
def loop_csv_local():

	conn = sqlite3.connect(project_db)
	
	for weather in weather_types:
		for i, area in enumerate(areas):

			# path to the previously downloaded .csv
			path = f"data/weather/{weather}/{area}_{weather}.csv"
	
			try:
				df = pd.read_csv(path,
								sep=',',					# Handle variable spacing
								skipinitialspace=True,		# Skip leading spaces
								skip_blank_lines=True,		# Skip empty lines
								skiprows=0,					# Skip the first 5 lines of metadata
								na_values=['---'],			# Specifies which value indicates null/missing data.
								keep_default_na=True,		# Retain default NA values.
								dtype={'year':int}
				)
	
				# Add an 'area' column to the DataFrame so this can be queried independently.
				df['area'] = area
	
				# Replace the table on the first area (clears old data),
				# then append for the rest — avoids duplicates on re-runs.
				write_mode = 'replace' if i == 0 else 'append'
	
				df.to_sql(
					name=weather,
					con=conn,
					if_exists=write_mode,
					index=False
				)
	
				# Write to console to monitor progress.
				print(f"Table {weather} / {area} updated.")
	
			# Write to console to show if any errors occur, but continue processing the rest of the data.
			except Exception as e:
				print(f"Error with updating table ({weather}/{area}): {str(e)}.")
	
	conn.close()
	
	print("All tables updated successfully.")

response = input("Do you wish to proceed with loading SQLite directly from locally saved .csv files?: y /n")

if response.lower() == 'y':
	print('Proceeding with download...')
	loop_csv_local()
	print('Database updated.')

elif response.lower() =='n':
	print('Action aborted.')

else:
	print('Incorrect input. Try again.')

#### 4.4.3 Test SQLite query

In [ ]:
conn = sqlite3.connect(project_db)

df_rainfall_all = pd.read_sql('SELECT * FROM Rainfall ', conn)
conn.close()

df_rainfall_all

# Exploratory Data Analysis for Crop and Weather Data

## [Contents](#contents)

- [1. Overview](#overview)
- [2. Environment Configuration](#2-environment-configuration)
- [3. Crop Yields](#3-crop-yields)
  - [3.1 UK Yearly Crop Yields](#31-uk-yearly-crop-yields)
  - [3.2 UK Country Wheat Crop Yields](#32-uk-country-wheat-crop-yields)
  - [3.3 England Regional Wheat Crop Yields](#33-england-region-wheat-crop-yields)
- [4. Combined Weather & Crop Yield Data Analysis](#4-combined-weather--crop-data-analysis)
  - [4.1 West Midlands Wheat Yield and Midlands Weather](#41-west-midlands-wheat-yield-and-midlands-weather)
  - [4.2 West Midlands Monthly Maximum Temperature and Wheat Yields](#42-west-midlands-monthly-maximum-temperature-and-wheat-yields)
  - [4.3 West Midlands Monthly Airfrost Days and Wheat Yields](#43-west-midlands-monthly-airfrost-days-and-wheat-yields)

## [2. Environment Configuration](#contents)

In [ ]:
%pip install scikit-learn

In [ ]:
import pandas as pd
import sqlite3
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LinearRegression
import numpy as np

In [ ]:
project_db  = 'data/project_db.db'

## [3. Crop Yields](#contents)

The UK has a varied landscape with varying soil, weather, and topographic conditions. These could all be factors in crop yields. To reduce the influence of these, yield data for just East Anglia will be analysed.

### [3.1 UK Yearly Crop Yields](#contents)

- What do Crop yields look like over the years?
- Is there variation that might be explained by weather variation across the years?
- Which crop shows most promise for analysis?

#### UK Yearly Crop Data

In [ ]:
qry_yields = """
	SELECT 
		o.Year,
		o."United Kingdom" AS UK_Oats,
		osr."United Kingdom" AS UK_OSR,
		sb."United Kingdom" AS UK_Spring_Barley,
		w."United Kingdom" AS UK_Wheat,
		wb."United Kingdom" AS UK_Winter_Barley

	FROM
		yields_oats o

		JOIN yields_OSR osr
			ON o.year = osr.year

		JOIN yields_spring_barley sb
			ON o.year = sb.year

		JOIN yields_wheat w
			ON o.year = w.year

		JOIN yields_winter_barley wb
			ON o.year = wb.year

	WHERE
		o.year >= 1998 AND o.year < 2026
"""

conn = sqlite3.connect(project_db)

df_yields = pd.read_sql(qry_yields, conn)

conn.close()

df_yields.head(10)

#### UK Yearly Crop Yields over Time Line Plot

In [ ]:
# define plot
plt.figure(figsize=(16,5))

# Create Month number field for x-axis and plot each year as a line on the same figure.
df_yields.loc[:, 'Year'] = pd.to_datetime(df_yields['Year'], format='%Y').dt.year
plt.plot(df_yields.Year, df_yields.UK_Oats , color='tab:red', label='Oats')
plt.plot(df_yields.Year, df_yields.UK_OSR , color='tab:blue', label='OSR')
plt.plot(df_yields.Year, df_yields.UK_Spring_Barley , color='tab:green', label='Spring Barley')
plt.plot(df_yields.Year, df_yields.UK_Wheat , color='tab:purple', label='Wheat')
plt.plot(df_yields.Year, df_yields.UK_Winter_Barley , color='tab:orange', label='Winter Barley')

# add figure elements
plt.title(f'UK Annual Crop Yields (1999 - 2025)')
plt.xlabel('Year')
plt.ylabel('Rainfall (mm)')
plt.legend(loc=2, ncols = 1)
plt.show()

Observations:

- Yields evidently vary from year to year.
- These variations appear to be consistent across crops.
- Wheat is the highest yielding crop.

#### UK Yearly Crop Yield Correlation Heatmap

A correlation Heatmap will show if the yields are correlated.

In [ ]:
# Drop Year column
df_yields_corr = df_yields.drop(columns=['Year'], inplace=False)
df_yields_corr.head()

In [ ]:
# Assuming 'data' is your DataFrame
corr = df_yields_corr.corr()

# Create a heatmap
plt.figure(figsize=(10, 8))
sns.heatmap(corr, annot=True, fmt=".2f", cmap='coolwarm', square=True)
plt.title('Correlation Heatmap')
plt.show()

Observations:

- Wheat has high correlations with Winter Barley, Spring Barley and Oats.
- Oil Seed Rape (OSR) yield has little correlation with any other crop.

This suggests some underlying cause for at least Wheat, Spring Barley and Oats yields to fluctuate together. This could well be weather related, or ALSO might be related to growing location.

### [3.2 UK Country Wheat Crop Yields](#contents)

- For a single crop, how does wheat yield differ across the UK Countries?
- Does yield across countries correlate, well, or do they show variation?

##### UK Countries Wheat Data

In [ ]:
qry_regional_wheat_yields = """
	SELECT 
		Year,
		England,
		Scotland,
		Wales,
		"Northern Ireland" AS NI

	FROM
		yields_wheat

	WHERE
		year >= 1998 AND year < 2026
"""

conn = sqlite3.connect(project_db)

df_wheat_yields = pd.read_sql(qry_regional_wheat_yields, conn)

conn.close()

df_wheat_yields.head(10)

##### UK Counties Wheat Plot

In [ ]:
# define plot
plt.figure(figsize=(16,5))

# Create Month number field for x-axis and plot each year as a line on the same figure.
df_wheat_yields.loc[:, 'Year'] = pd.to_datetime(df_wheat_yields['Year'], format='%Y').dt.year
plt.plot(df_wheat_yields.Year, df_wheat_yields.England , color='tab:orange', label='England')
plt.plot(df_wheat_yields.Year, df_wheat_yields.Scotland , color='tab:blue', label='Scotland')
plt.plot(df_wheat_yields.Year, df_wheat_yields.Wales , color='tab:red', label='Wales')
plt.plot(df_wheat_yields.Year, df_wheat_yields.NI , color='tab:green', label='Northern Ireland')


# add figure elements
plt.title(f'UK Annual Wheat Yields Across Regions (1999 - 2025)')
plt.xlabel('Year')
plt.ylabel('Wheat Yield')
plt.legend(loc=2, ncols = 1)
plt.show()

##### UK Counties Wheat Yield Correlation Heatmap

In [ ]:
# Drop Year column
df_wheat_yields_corr = df_wheat_yields.drop(columns=['Year'], inplace=False)
df_wheat_yields_corr.head()

In [ ]:
# Assuming 'data' is your DataFrame
corr = df_wheat_yields_corr.corr()

# Create a heatmap
plt.figure(figsize=(10, 8))
sns.heatmap(corr, annot=True, fmt=".2f", cmap='coolwarm', square=True)
plt.title('Correlation Heatmap')
plt.show()

Observations:

- Correlation in wheat yield between the UK counties is not very high, with the best being between Wales and England.
- This indicates likely differences between these countries that may be related to weather or other factors such as soil quality, farming practices or topology.
- For an assessment of weather impact on crop yields, this needs to be minimised, with areas that have as consistent variation as possible.

### [3.3 England Region Wheat Crop Yields](#contents)

- For a single crop, how does the wheat yield differ across England Regions?
- Does yield across England regions correlate?

##### Regional England Wheat Yield Data

In [ ]:
qry_england_wheat_yields = """
	SELECT 
		Year,
		"North East" AS NE,
		"North West and Merseyside" AS NW,
		"Yorkshire & The Humber" AS Y,
		"East Midlands" AS EM,
		"West Midlands" AS WM,
		"Eastern" AS E,
		"South East and London" AS SE,
		"South West" AS SW

	FROM
		yields_wheat

	WHERE
		year >= 1998 AND year < 2026
"""

conn = sqlite3.connect(project_db)

df_wheat_yields_england = pd.read_sql(qry_england_wheat_yields, conn)

conn.close()

df_wheat_yields_england.head(10)

##### Regional England Wheat Yield Plot

In [ ]:
# define plot
plt.figure(figsize=(16,5))

# Create Month number field for x-axis and plot each year as a line on the same figure.
df_wheat_yields_england.loc[:, 'Year'] = pd.to_datetime(df_wheat_yields_england['Year'], format='%Y').dt.year
plt.plot(df_wheat_yields_england.Year, df_wheat_yields_england.NE , color='tab:blue', label='North East')
plt.plot(df_wheat_yields_england.Year, df_wheat_yields_england.NW , color='tab:orange', label='North West & Merseyside')
plt.plot(df_wheat_yields_england.Year, df_wheat_yields_england.Y , color='tab:green', label='Yorkshire & The Humber')
plt.plot(df_wheat_yields_england.Year, df_wheat_yields_england.EM , color='tab:red', label='East Midlands')
plt.plot(df_wheat_yields_england.Year, df_wheat_yields_england.WM , color='tab:purple', label='West Midlands')
plt.plot(df_wheat_yields_england.Year, df_wheat_yields_england.E , color='tab:brown', label='Eastern')
plt.plot(df_wheat_yields_england.Year, df_wheat_yields_england.SE , color='tab:pink', label='South East & London')
plt.plot(df_wheat_yields_england.Year, df_wheat_yields_england.SW , color='tab:grey', label='South West')


# add figure elements
plt.title(f'Regional England Wheat Yields (1999 - 2025)')
plt.xlabel('Year')
plt.ylabel('Wheat Yield')
plt.legend(loc=2, ncols = 3)
plt.show()

##### Regional England Wheat Yield Correlation Heatmap

In [ ]:
# Drop Year column
df_wheat_yields_england_corr = df_wheat_yields_england.drop(columns=['Year'], inplace=False)
df_wheat_yields_england_corr.head()

In [ ]:
# Assuming 'data' is your DataFrame
corr = df_wheat_yields_england_corr.corr()

# Create a heatmap
plt.figure(figsize=(10, 8))
sns.heatmap(corr, annot=True, fmt=".2f", cmap='coolwarm', square=True)
plt.title('Correlation Heatmap')
plt.show()

Observations:

- The North West has the lowest correlation with all other regions. This is apparent when looking at yield over the years from 1999 to 2025, much lower than all other regions.
- The highest correlations are in adjacent regions, such as South West and South East & London, West and East Midlands, or even South East & London with West Midlands and Yorkshire & The Humber.
- There are clearly regional influences in play. To reduce this noise and to identify the influence of weather factors, selecting a single region to focus on would be beneficial.

## [4. Combined Weather & Crop Data Analysis](#contents)

### [4.1 West Midlands Wheat Yield and Midlands Weather](#contents)

- Annual Wheat Yield for West Midlands
- Annual averages for each weather
- Correlation Heatmap

#### Data for West Midlands Wheat Yield and Annual Weather Averages

In [ ]:
query_wm_annual = """
	SELECT

		w."West Midlands" AS wm_wheat_yield,

		a.ann AS wm_airfrost,
		rd.ann AS wm_raindays,
		r.ann AS wm_rainfall,
		s.ann AS wm_sunshine,
		tx.ann AS wm_max_temp,
		tm.ann AS wm_mean_temp,
		tn.ann AS wm_min_temp
		

	FROM yields_wheat w

		JOIN Airfrost a ON w."Year" = a.year
		JOIN Rainfall r ON w."Year" = r.year
		JOIN Raindays1mm rd ON w."Year" = rd.year
		JOIN Sunshine s ON w."Year" = s.year
		JOIN Tmax tx ON w."Year" = tx.year
		JOIN Tmean tm ON w."Year" = tm.year
		JOIN Tmin tn ON w."Year" = tn.year
	
	WHERE
		a.area = 'Midlands'
		AND r.area = 'Midlands'
		AND rd.area = 'Midlands'
		AND s.area = 'Midlands'
		AND tx.area = 'Midlands'
		AND tm.area = 'Midlands'
		AND tn.area = 'Midlands'
		AND s.area = 'Midlands'
		AND w.year >= 1999 AND w.year <= 2025
"""

conn = sqlite3.connect(project_db)

df_annual_wm_wheat = pd.read_sql(query_wm_annual, conn)

conn.close()

df_annual_wm_wheat.head()

#### West Midlands Wheat Yield Weather Correlation Heatmap

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

# Assuming 'data' is your DataFrame
corr = df_annual_wm_wheat.corr()

# Create a heatmap
plt.figure(figsize=(10, 8))
sns.heatmap(corr, annot=True, fmt=".2f", cmap='coolwarm', square=True)
plt.title('West Midlands Wheat Yield Weather Correlation Heatmap')
plt.show()

Observations:

- West Midlands Wheat Yield does not have any high correlations with any annual weather values.
- It may be that averaging over the year hides more granular weather factors, since wheat will only be influenced by the weather over the production cycle.
- The highest positive correlations are for annual monthly average maximum temperature and annual monthly average temperature.
- The highest negative correlation was for annual average monthly airfrost days.

### [4.2 West Midlands Monthly Maximum Temperature and Wheat Yields](#contents)

- Annual Wheat Yields for West Midlands
- Monthly Maximum Temperature
- Correlation Heatmap

##### West Midlands Monthly Max Temp and Wheat Yield Data

In [ ]:
query_wm_monthly_max_temp = """
	SELECT
		w."West Midlands" AS wm_wheat_yield,
		tx.jan,
		tx.feb,
		tx.mar,
		tx.apr,
		tx.may,
		tx.jun,
		tx.jul,
		tx.aug,
		tx.sep,
		tx.oct,
		tx.nov,
		tx.dec

	FROM yields_wheat w

		JOIN Tmax tx ON w."Year" = tx.year
	
	WHERE
		tx.area = 'Midlands'
		AND w.year >= 1999 AND w.year <= 2025
"""

conn = sqlite3.connect(project_db)

df_wm_monthly_max_temp_wheat_yield = pd.read_sql(query_wm_monthly_max_temp, conn)

conn.close()

df_wm_monthly_max_temp_wheat_yield.head()

##### Correlation Heatmap

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

# Assuming 'data' is your DataFrame
corr = df_wm_monthly_max_temp_wheat_yield.corr()

# Create a heatmap
plt.figure(figsize=(10, 8))
sns.heatmap(corr, annot=True, fmt=".2f", cmap='coolwarm', square=True)
plt.title('West Midlands Monthly Max Temp Wheat Yield Correlation Heatmap')
plt.show()

Observations:

- None of the months show a high correlation for Max Temperature with West Midlands Wheat Yield.
- The highest correlation is for the Month of February.

#### Scatter Plot

In [ ]:
# Remove unwanted columns
df_wm_march_max_temp_wheat_yield = df_wm_monthly_max_temp_wheat_yield.drop(columns=['jan','mar','apr','may','jun','jul','aug','sep','oct','nov','dec'])
df_wm_march_max_temp_wheat_yield.head()

In [ ]:
# Define the arrays for the linear regression model.
x_uk_max_temp = np.array(df_wm_march_max_temp_wheat_yield['feb']).reshape(-1, 1)
y_wm_w_yield = np.array(df_wm_march_max_temp_wheat_yield['wm_wheat_yield']).reshape(-1, 1)

# Create the model.
model = LinearRegression()

# Fit the model to the data
model.fit(x_uk_max_temp, y_wm_w_yield)

r_sq = model.score(x_uk_max_temp, y_wm_w_yield)
print(f"Coefficient of determination: {r_sq}")
print(f"Intercept: {model.intercept_}")
print(f"Coefficient: {model.coef_}")

# Make model predictions
y_pred = model.predict(x_uk_max_temp)

# Plot the arrays and results of the model prediction
plt.scatter(x_uk_max_temp, y_wm_w_yield, color='blue', label='Actual Values')
plt.plot(x_uk_max_temp, y_pred, color='red', label='Model Fitted Line')
plt.title('West Midlands Wheat Yield vs February Maximum Daily Temperature')
plt.xlabel('Max Monthly Temperature')
plt.ylabel('WM Wheat Yield')
plt.legend()
plt.show()

Also a correlation can be calculated between the February Midlands Monthly Max Temperature and West Midlands Wheat Yield.

In [ ]:
max_t_yield_corr = df_wm_march_max_temp_wheat_yield['feb'].corr(df_wm_march_max_temp_wheat_yield['wm_wheat_yield'])
max_t_yield_corr

Observation:

- There is a reasonable correlation coefficient (R = 0.070) between Maximum daily temperature and wheat yield in the West Midlands.
- However, there is only a very weak correlation between Midland Maximum Temperature in February and West Midlands Wheat Yield.

### [4.3 West Midlands Monthly Airfrost Days and Wheat Yields](#contents)

- Annual Wheat Yields for West Midlands
- Monthly Airfrost Days
- Correlation Heatmap

##### West Midlands Monthly Airfrost Days and Wheat Yield Data

In [ ]:
query_wm_monthly_airfrost = """
	SELECT
		w."West Midlands" AS wm_wheat_yield,
		a.jan,
		a.feb,
		a.mar,
		a.apr,
		a.may,
		a.jun,
		a.jul,
		a.aug,
		a.sep,
		a.oct,
		a.nov,
		a.dec

	FROM yields_wheat w

		JOIN AirFrost a ON w."Year" = a.year
	
	WHERE
		a.area = 'Midlands'
		AND w.year >= 1999 AND w.year <= 2025
"""

conn = sqlite3.connect(project_db)

df_wm_monthly_airfrost_wheat_yield = pd.read_sql(query_wm_monthly_airfrost, conn)

conn.close()

df_wm_monthly_airfrost_wheat_yield.head()

##### Correlation Heatmap

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

# Assuming 'data' is your DataFrame
corr = df_wm_monthly_airfrost_wheat_yield.corr()

# Create a heatmap
plt.figure(figsize=(10, 8))
sns.heatmap(corr, annot=True, fmt=".2f", cmap='coolwarm', square=True)
plt.title('West Midlands Monthly Max Temp Wheat Yield Correlation Heatmap')
plt.show()

Observations:

- There are no especially strong monthly correlations for Airfrost days for Wheat Yield in the West Midlands.
- The most negatively correlated months are March (-0.35) and May (-0.34), with March marginally more impactful.

#### Scatter Plot

In [ ]:
df_wm_march_airfrost_wheat_yield = df_wm_monthly_airfrost_wheat_yield.drop(columns=['jan','feb','apr','may','jun','jul','aug','sep','oct','nov','dec'])
df_wm_march_airfrost_wheat_yield.head()

In [ ]:
# Define the arrays for the linear regression model.
x_mid_airfrost = np.array(df_wm_march_airfrost_wheat_yield['mar']).reshape(-1, 1)
y_wm_w_yield = np.array(df_wm_march_airfrost_wheat_yield['wm_wheat_yield']).reshape(-1, 1)

# Create the model.
model = LinearRegression()

# Fit the model to the data
model.fit(x_mid_airfrost, y_wm_w_yield)

r_sq = model.score(x_mid_airfrost, y_wm_w_yield)
print(f"Coefficient of determination: {r_sq}")
print(f"Intercept: {model.intercept_}")
print(f"Coefficient: {model.coef_}")

# Make model predictions
y_pred = model.predict(x_mid_airfrost)

# Plot the arrays and results of the model prediction
plt.scatter(x_mid_airfrost, y_wm_w_yield, color='blue', label='Actual Values')
plt.plot(x_mid_airfrost, y_pred, color='red', label='Model Fitted Line')
plt.title('West Midlands Wheat Yield vs Airfrost Days')
plt.xlabel('March Airfrost days')
plt.ylabel('WM Wheat Yield')
plt.legend()
plt.show()

Also a correlation can be calculated between the March Midlands Monthly Airfrost Days and West Midlands Wheat Yield.

In [ ]:
frost_yield_corr = df_wm_march_airfrost_wheat_yield['mar'].corr(df_wm_march_airfrost_wheat_yield['wm_wheat_yield'])
frost_yield_corr

Observation:

- The regression is a poor fit (0.12) between monthly airfrost days in March and Wheat Yield in the West Midlands.
- There is only a weak negative correlation of -0.35 between Midlands airfrost days in March and West Midlands Wheat Yield.